In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())

CUDA available: True
CUDA devices: 1
Current device: 0


In [10]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Paths to data
image_dir = "../1_DatasetCharacteristics/seamounts_seg_cropped"
csv_path = "../1_DatasetCharacteristics/merged_pixel_coordinates.csv"

# Constants
IMAGE_SIZE = (256, 256)  # Resize images to 256x256
BATCH_SIZE = 32
EPOCHS = 20

# Load bounding box data
bbox_data = pd.read_csv(csv_path)

# Function to load and preprocess images with added checks
def load_image(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)  # Assuming grayscale images
    if image is None:
        print(f"Error loading image: {image_path}")
        return None
    image = cv2.resize(image, IMAGE_SIZE)  # Resize to consistent dimensions
    print(f"Loaded image shape: {image.shape}")
    image = image / 255.0  # Normalize pixel values to [0, 1]
    return image

# Function to preprocess bounding boxes with added checks
def preprocess_bboxes(row, image_shape):
    x_min, y_min, x_max, y_max = row[['x_min', 'y_min', 'x_max', 'y_max']]
    height, width = image_shape[:2]
    print(f"Image shape: {image_shape}, Bounding box: {x_min}, {y_min}, {x_max}, {y_max}")
    # Normalize coordinates to [0, 1]
    x_min /= width
    y_min /= height
    x_max /= width
    y_max /= height
    return [x_min, y_min, x_max, y_max]

# Prepare dataset
images = []
bboxes = []
for _, row in bbox_data.iterrows():
    img_path = os.path.join(image_dir, row['image_name'])
    if os.path.exists(img_path):
        # Load image
        image = load_image(img_path)
        if image is not None:
            images.append(image)
            # Infer original size from the loaded image
            original_size = image.shape
            # Preprocess bounding box
            bbox = preprocess_bboxes(row, original_size)
            bboxes.append(bbox)

images = np.array(images)
bboxes = np.array(bboxes)

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(images, bboxes, test_size=0.2, random_state=42)

# Add channel dimension (for grayscale images)
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]

# Define the model
def create_model():
    inputs = layers.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 1))
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(4, activation='sigmoid')(x)  # 4 outputs for bounding box coordinates
    model = models.Model(inputs, outputs)
    return model

# Create and compile the model
model = create_model()
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# Evaluate model performance
val_loss, val_mae = model.evaluate(X_val, y_val)
print(f"Validation Loss: {val_loss}, Validation MAE: {val_mae}")

# Visualize predictions
def plot_predictions(images, true_bboxes, pred_bboxes):
    for i in range(5):  # Show 5 examples
        img = images[i].squeeze()
        true_bbox = true_bboxes[i]
        pred_bbox = pred_bboxes[i]
        assert np.all(pred_bbox >= 0) and np.all(pred_bbox <= 1), "Predicted bounding box out of bounds"
        print(f"True bbox: {true_bbox}, Predicted bbox: {pred_bbox}")
        
        plt.imshow(img, cmap='gray')
        # Plot true bounding box
        h, w = img.shape
        true_rect = [
            true_bbox[0] * w, true_bbox[1] * h,
            (true_bbox[2] - true_bbox[0]) * w, (true_bbox[3] - true_bbox[1]) * h
        ]
        plt.gca().add_patch(plt.Rectangle(
            (true_rect[0], true_rect[1]),
            true_rect[2], true_rect[3],
            edgecolor='green', facecolor='none', lw=2, label='True'
        ))
        # Plot predicted bounding box
        pred_rect = [
            pred_bbox[0] * w, pred_bbox[1] * h,
            (pred_bbox[2] - pred_bbox[0]) * w, (pred_bbox[3] - pred_bbox[1]) * h
        ]
        plt.gca().add_patch(plt.Rectangle(
            (pred_rect[0], pred_rect[1]),
            pred_rect[2], pred_rect[3],
            edgecolor='red', facecolor='none', lw=2, label='Predicted'
        ))
        plt.legend()
        plt.show()

# Make predictions on the validation set
pred_bboxes = model.predict(X_val)

# Plot predictions vs true bounding boxes
plot_predictions(X_val, y_val, pred_bboxes)


Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 136, 89, 263, 268
Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 153, 118, 245, 238
Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 137, 99, 262, 258
Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 166, 142, 232, 214
Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 134, 104, 265, 252
Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 139, 112, 259, 245
Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 137, 110, 262, 247
Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 139, 112, 260, 244
Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 143, 119, 255, 238
Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 147, 124, 252, 232
Loaded image shape: (256, 256)
Image shape: (256, 256), Bounding box: 131, 106, 267, 250
Loaded image shape: (25